# Mini Project: Indian Stock Market Analysis & Future Return Prediction (2000 - 2020)
### Course: Machine Learning (LP-3)
**Problem Statement:** Use Indian market dataset from 2000 to 2020 to analyze ups and downs in the market and predict future stock price returns.
**Dataset Link:** https://www.kaggle.com/datasets/sagara9595/stock-data

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score

print("Libraries imported successfully.")


Libraries imported successfully.


In [2]:
df = pd.read_csv("indian_stock_market_2000_2020.csv")
print(f"Total Trading Days: {len(df)} records")
print(f"Date Range: {df['Date'].iloc[0]} to {df['Date'].iloc[-1]}")
print(f"Columns: {list(df.columns)}")
df.head()


Total Trading Days: 5479 records
Date Range: 2000-01-03 to 2020-12-31
Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume']


In [3]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / (loss + 1e-9)
    return 100 - (100 / (1 + rs))

df["Return_1d"] = df["Close"].pct_change()
df["Return_5d"] = df["Close"].pct_change(5)
df["SMA_10"] = df["Close"].rolling(10).mean()
df["SMA_50"] = df["Close"].rolling(50).mean()
df["SMA_Ratio"] = df["Close"] / df["SMA_50"]
df["Volatility_10"] = df["Return_1d"].rolling(10).std()
df["HL_Spread"] = (df["High"] - df["Low"]) / df["Close"]
df["RSI"] = compute_rsi(df["Close"], period=14)

# Future Return & Trend Targets
df["Future_Return_5D"] = (df["Close"].shift(-5) - df["Close"]) / df["Close"]
df["Market_Trend"] = (df["Future_Return_5D"] > 0).astype(int)

df = df.dropna().reset_index(drop=True)
print("Engineered features: Return_1d, Return_5d, SMA_10, SMA_50, SMA_Ratio, Volatility_10, HL_Spread, RSI
Cleaned shape:", df.shape)


Engineered features: Return_1d, Return_5d, SMA_10, SMA_50, SMA_Ratio, Volatility_10, HL_Spread, RSI
Cleaned shape: (5425, 16)


In [4]:
feature_cols = ["Return_1d", "Return_5d", "SMA_Ratio", "Volatility_10", "HL_Spread", "RSI"]
X = df[feature_cols]
y_class = df["Market_Trend"]
y_reg = df["Future_Return_5D"]

X_train, X_test, y_cls_train, y_cls_test, y_reg_train, y_reg_test = train_test_split(
    X, y_class, y_reg, test_size=0.2, random_state=42, stratify=y_class
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Model 1: Classification (Ups & Downs)
clf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
clf.fit(X_train_s, y_cls_train)
pred_cls = clf.predict(X_test_s)

print(f"Market Direction Accuracy: {accuracy_score(y_cls_test, pred_cls) * 100:.2f}%")
print("
Classification Report:
", classification_report(y_cls_test, pred_cls, target_names=["Down (Bearish)", "Up (Bullish)"]))


Market Direction Accuracy: 77.97%

Classification Report:
                precision    recall  f1-score   support

Down (Bearish)       0.74      0.76      0.75       467
  Up (Bullish)       0.81      0.80      0.80       618

      accuracy                           0.78      1085


In [5]:
# Model 2: Regression (Return magnitude)
reg = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42)
reg.fit(X_train_s, y_reg_train)
pred_reg = reg.predict(X_test_s)

mse = mean_squared_error(y_reg_test, pred_reg)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_reg_test, pred_reg)
r2 = r2_score(y_reg_test, pred_reg)

print("Regression Evaluation (Future 5-Day Return):")
print(f"• MSE:  {mse:.6f}")
print(f"• RMSE: {rmse:.4f}")
print(f"• MAE:  {mae:.4f} ({mae * 100:.2f}% average deviation)")
print(f"• R2:   {r2:.4f}")


Regression Evaluation (Future 5-Day Return):
• MSE:  0.001193
• RMSE: 0.0345
• MAE:  0.0280 (2.80% average deviation)
• R2:   0.5036
